In [0]:
# Install Spacy and the model AS A PACKAGE directly
%pip install spacy==3.7.2
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl


dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.8/818.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 149.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.0 MB/s eta 0:00:00
  Attempting uninstall: smart-open
    Found existing installation: smart_open 7.5.0
    Uninstalling smart_open-7.5.0:
      Successfully uninstalled smart_open-7.5.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-082aed7d-6e16-43b7-827f-c4331728c6f4
    Can't uninstall 'numpy'. No files were found to uninstall.
  Attempting uninstall: cloudpathlib
    Found existing installation: cloudpathlib 0.23.0
    Uninstalling cloudpathlib-0.23.0:
      Successfully uninstalled cloudpathlib-0.23.0
  Attempting uninsta

In [0]:
import spacy
import pandas as pd
from pyspark.sql.functions import pandas_udf, col, PandasUDFType
from pyspark.sql.types import StringType
import re

# Initialize Spacy on the driver to verify (Workers will load it inside the UDF)
nlp_test = spacy.load("en_core_web_sm")
print("NLP Model Loaded Successfully.")

NLP Model Loaded Successfully.


In [0]:
import spacy
import pandas as pd
from pyspark.sql.functions import pandas_udf, col, PandasUDFType
from pyspark.sql.types import StringType
import re
# IMPORT THE MODEL AS A MODULE
import en_core_web_sm

@pandas_udf(StringType())
def extract_core_product_name(titles: pd.Series) -> pd.Series:
    # Load from the imported module
    nlp = en_core_web_sm.load(disable=["ner", "parser", "lemmatizer"])
    nlp.enable_pipe("tagger")
    nlp.enable_pipe("attribute_ruler")

    cleaned_titles = []
    
    for doc in nlp.pipe(titles.astype(str), batch_size=50, n_process=1):
        relevant_tokens = []
        
        # Clean brackets
        text_clean = re.sub(r'\[.*?\]|\(.*?\)', '', doc.text)
        if len(text_clean) != len(doc.text):
            doc = nlp.make_doc(text_clean)
            doc = nlp(doc)

        count_tokens = 0
        for token in doc:
            if token.text in ["-", ",", "|", ":", ";", "/"]:
                break
            # Keep Brands (PROPN), Nouns (NOUN) and leading Adjectives (ADJ)
            if token.pos_ in ["PROPN", "NOUN"] or (token.pos_ == "ADJ" and count_tokens < 1):
                relevant_tokens.append(token.text)
                count_tokens += 1
            if count_tokens >= 3:
                break
        
        if not relevant_tokens:
            cleaned_titles.append(" ".join(doc.text.split()[:2]))
        else:
            cleaned_titles.append(" ".join(relevant_tokens))
            
    return pd.Series(cleaned_titles)

In [0]:
from pyspark.sql.functions import col, coalesce, count, row_number, desc
from pyspark.sql.window import Window

# --- 1. LOAD DATA ---
INPUT_PATH = "/Volumes/workspace/default/meta/6fc66a73-5532-4e90-92fc-5cc6c100d8a4.csv" 
df = spark.read.csv(INPUT_PATH, header=True)

print(f"Initial Row Count: {df.count()}")

# --- 2. APPLY NLP (Create 'product_title') ---
print("Running NLP Extraction...")
df_nlp = df.withColumn("product_title", extract_core_product_name(col("title")))

# --- 3. IMPUTE MAIN_CATEGORY (Fill Nulls) ---
print("Imputing Missing Main Categories...")

# Step A: Create a 'Lookup Table' of the most frequent main_category per category
category_stats = df_nlp.filter(col("main_category").isNotNull()) \
    .groupBy("category", "main_category") \
    .agg(count("*").alias("cnt"))

# Step B: Window function to get the #1 most frequent main_category
window_spec = Window.partitionBy("category").orderBy(col("cnt").desc())

mode_mapping = category_stats.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .select(col("category"), col("main_category").alias("mode_main_cat"))

# Step C: Join back to original data
df_joined = df_nlp.join(mode_mapping, on="category", how="left")

# Step D: Fill Nulls
df_final = df_joined.withColumn("main_category", coalesce(col("main_category"), col("mode_main_cat"))) \
    .drop("mode_main_cat")  # Cleanup helper column

# --- 4. VERIFY RESULTS ---
display(df_final.select("title", "product_title", "category", "main_category").limit(40))

Initial Row Count: 1382399
Running NLP Extraction...
Imputing Missing Main Categories...


title,product_title,category,main_category
"Cleaning Gel for Automotive Dust Car Crevice Cleaner Air Cent Interior Detail Removal Putty Cleaning Keyboard,Cleaning Putty for Car *1",Gel Automotive Dust,null,Musical Instruments
NEW TURNTABLE STYLUS FOR ION TTUSB TTUSB05 TTUSB10 LPDOCK,NEW STYLUS ION,null,Musical Instruments
CynKen 4PCS Black Nickel Speaker Spikes Pad 5x25mm,CynKen 4PCS Black,null,Musical Instruments
"Valentine Gifts Ideas Tibetan Singing Bowl Set Black - Om Mani Padme Hum - Includes 4 inch Singing Bowl, Cushion & Mallet - For Healing Meditation Prayer and Yoga |Handmade| (OM-Black)",Valentine Gifts Ideas,null,Musical Instruments
KENPMA Adjustable Desktop Tripod Microphone Stand Table Top Mic Holder with Clip Mount,KENPMA Desktop Tripod,Musical Instruments,Musical Instruments
"Music Stand, Professional Music Conductor Stand Sheet/Book Tripod Holder with Two Carrying Bags Lightweight Suitable for Violin, Guitar, Flute and Instrumental Performance (original version)",Music Stand,Musical Instruments,Musical Instruments
HOEREV 5.8GHz Electric Guitar Accessories Wireless Guitar Bass Stand Transmitter Receiver System 4 Channels for Electric Bass Cordless Amplifier Guitar Cable Jack With Rechargeable Lithium Battery,HOEREV Electric Guitar,Musical Instruments,Musical Instruments
FLEOR 2pcs Full Size 15mm Shaft Guitar Volume Pot A500K Audio Taper Guitar Potentiometer with Bayonet,FLEOR Full Size,Musical Instruments,Musical Instruments
"Oscar Schmidt OU7TE Spalted Mango Tenor Acoustic/Electric Ukulele Aquila String, True Tune Tuner Package",Oscar Schmidt OU7TE,Musical Instruments,Musical Instruments
Vivarium Electronics VE-100 Thermostat for Snake habitats Bundle with Carolina Custom Cages' Chlorhexidine Solution 2%; 1 Refill Makes 32 oz. of Working Solution,Vivarium Electronics VE-100,Musical Instruments,Musical Instruments


In [0]:
# Define Output Path
OUTPUT_PATH = "/Volumes/workspace/default/meta/updated_titles_parquet"
df_final.write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"SUCCESS: Data saved to {OUTPUT_PATH}")

SUCCESS: Data saved to /Volumes/workspace/default/meta/updated_titles_parquet


In [0]:
from pyspark.sql.functions import col
INPUT_PATH = "/Volumes/workspace/default/meta/updated_titles_parquet"
df = spark.read.parquet(INPUT_PATH)

# 2. Drop the old messy 'title' and rename 'product_title' to 'title'
df_final = df.drop("title").withColumnRenamed("product_title", "title")
FINAL_OUTPUT_PATH = "/Volumes/workspace/default/meta/product_metadata_final"

df_final.write.mode("overwrite").parquet(FINAL_OUTPUT_PATH)

print(f"SUCCESS: Final dataset saved to {FINAL_OUTPUT_PATH}")

display(df_final.select("parent_asin", "title", "main_category").limit(5))

SUCCESS: Final dataset saved to /Volumes/workspace/default/meta/product_metadata_final


parent_asin,title,main_category
B07NZB4CYF,SLIMBELLE Sauna Suit,AMAZON FASHION
B0BDGYW27L,Lands End Womens,AMAZON FASHION
B082GPW2M8,Oriental Pearl Funny,AMAZON FASHION
B06XKYKP9J,FARYSAYS Women Sexy,AMAZON FASHION
B07RFQ873X,Girls Unicorn Dress,AMAZON FASHION


In [0]:



INPUT_PATH = "/Volumes/workspace/default/meta/product_metadata_final"
df = spark.read.parquet(INPUT_PATH)
all_columns = df.columns

# Remove 'parent_asin' and 'title' from the list temporarily
remaining_columns = [c for c in all_columns if c not in ['parent_asin', 'title']]

# Create the new ordered list: parent_asin -> title -> everything else
new_column_order = ['parent_asin', 'title'] + remaining_columns

# 3. Select columns in the new order and Coalesce to 1
df_reordered = df.select(new_column_order)

# 4. Write to Single File Path
SINGLE_FILE_PATH = "/Volumes/workspace/default/meta/product_metadata_single_ordered"

df_reordered.coalesce(1).write.mode("overwrite").parquet(SINGLE_FILE_PATH)

print(f"SUCCESS: Data saved to {SINGLE_FILE_PATH}")
print("Columns are now ordered as:", df_reordered.columns[:5]) # Verify first 5 columns

SUCCESS: Data saved to /Volumes/workspace/default/meta/product_metadata_single_ordered
Columns are now ordered as: ['parent_asin', 'title', 'category', 'main_category', 'brand']


In [0]:

FILE_PATH = "/Volumes/workspace/default/meta/product_metadata_single_ordered/part-00000-tid-8720867345796894793-fc9a87de-663c-4523-8a2e-113b71a29b2d-225-1.c000.snappy.parquet"

df_check = spark.read.parquet(FILE_PATH)

row_count = df_check.count()
col_count = len(df_check.columns)

print(f"Total Rows: {row_count}")
print(f"Total Columns: {col_count}")
print("\n--- Schema ---")
df_check.printSchema()

print("\n--- First 5 Rows (Check if 'title' is 2nd column) ---")
display(df_check.limit(5))

Total Rows: 1382399
Total Columns: 20

--- Schema ---
root
 |-- parent_asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- store: string (nullable = true)
 |-- price: string (nullable = true)
 |-- average_rating: string (nullable = true)
 |-- rating_number: string (nullable = true)
 |-- date_first_available: string (nullable = true)
 |-- total_reviews: string (nullable = true)
 |-- avg_rating: string (nullable = true)
 |-- rating_volatility: string (nullable = true)
 |-- verified_reviews: string (nullable = true)
 |-- unverified_reviews: string (nullable = true)
 |-- verified_5_star_pct: string (nullable = true)
 |-- unverified_5_star_pct: string (nullable = true)
 |-- reviews_with_helpful_votes: string (nullable = true)
 |-- avg_review_length: string (nullable = true)


--- First 5 Rows (Check if 'tit

parent_asin,title,category,main_category,brand,manufacturer,store,price,average_rating,rating_number,date_first_available,total_reviews,avg_rating,rating_volatility,verified_reviews,unverified_reviews,verified_5_star_pct,unverified_5_star_pct,reviews_with_helpful_votes,avg_review_length
B07NZB4CYF,SLIMBELLE Sauna Suit,null,AMAZON FASHION,Unbranded,null,SLIMBELLE,null,2.9,6,2019-04-10,1,4.0,null,1,0,0.0,0.0,0,162.0
B0BDGYW27L,Lands End Womens,null,AMAZON FASHION,Unbranded,null,Lands' End,null,5.0,1,2022-09-02,1,5.0,null,1,0,1.0,0.0,0,89.0
B082GPW2M8,Oriental Pearl Funny,null,AMAZON FASHION,Unbranded,null,Oriental Pearl,null,4.3,21,2019-12-20,1,5.0,null,1,0,1.0,0.0,0,28.0
B06XKYKP9J,FARYSAYS Women Sexy,null,AMAZON FASHION,Unbranded,null,FARYSAYS,null,3.0,6,2017-04-20,1,4.0,null,1,0,0.0,0.0,1,399.0
B07RFQ873X,Girls Unicorn Dress,null,AMAZON FASHION,Unbranded,null,LEMONBABY,null,4.9,11,2019-05-06,2,4.5,0.71,2,0,0.5,0.0,0,126.0
